# SAP E-commerce Order Fulfillment & Delivery Analytics Pipeline
**Case Study: ShopX — SAP ERP -> Data Warehouse -> Power BI**

### Pipeline Phases:
1. Install dependencies
2. MySQL connection & database setup
3. Extract SAP data from Excel (simulating SAP ERP flat-file export)
4. Data Validation (completeness, referential integrity, format, uniqueness)
5. Data Transformation & Feature Engineering
6. Star Schema DDL — create DW tables
7. ETL Load — staging -> warehouse tables
8. Power BI views + KPI summary table (MySQL-connected, no CSVs)
9. Unit Test Results
10. Power BI connection guide + DAX measures

## Cell 1 — Install Dependencies (run once)

In [28]:
import subprocess, sys
for pkg in ['sqlalchemy', 'pymysql', 'openpyxl', 'pandas']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'])
print('Dependencies ready')

Dependencies ready


## Cell 2 — Configuration
> Edit MySQL credentials and SAP Excel file path before running.

In [29]:
# MySQL credentials
DB_HOST     = '127.0.0.1'
DB_PORT     = 3306
DB_USER     = 'root'
DB_PASSWORD = 'Monika#80'  # change this
DB_NAME     = 'shopx_dw'   # will be created automatically

# Source file — must be in the same folder as this notebook
SAP_EXCEL = 'SAP-DataSet.xlsx'

print(f'Target DB : {DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}')
print(f'SAP File  : {SAP_EXCEL}')

Target DB : root@127.0.0.1:3306/shopx_dw
SAP File  : SAP-DataSet.xlsx


## Cell 3 — MySQL Connection & Database Setup

In [30]:
from sqlalchemy import create_engine, text

# Connect without DB to create it if missing
_root = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/', echo=False)
with _root.connect() as conn:
    conn.execute(text(f'CREATE DATABASE IF NOT EXISTS {DB_NAME}'))
    conn.commit()
print(f'Database {DB_NAME} ready')

engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
    pool_pre_ping=True, pool_size=5, max_overflow=2, echo=False
)
with engine.connect() as conn:
    conn.execute(text('SELECT 1'))
print('Connected to MySQL successfully')

Database shopx_dw ready
Connected to MySQL successfully


## Cell 4 — Extract SAP Data from Excel

**SAP ERP Source Tables mapped to Excel sheets:**
- `KNA1` -> Customer Master Data
- `LFA1` -> Vendor/Carrier Master Data
- `VBAK` -> Sales Order Header
- `VBAP` -> Sales Order Item
- `LIKP` -> Delivery Header
- `LIPS` -> Delivery Item
- `VTTK` -> Shipment Header
- `VTTP` -> Shipment Item

In [ ]:
import pandas as pd

xl = pd.ExcelFile(SAP_EXCEL)
print(f'Sheets found: {xl.sheet_names}')

kna1 = xl.parse('KNA1')
lfa1 = xl.parse('LFA1')
vbak = xl.parse('VBAK')
vbap = xl.parse('VBAP')
likp = xl.parse('LIKP')
lips = xl.parse('LIPS')
vttk = xl.parse('VTTK')
vttp = xl.parse('VTTP')

def clean_cols(df):
    df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
    return df

kna1=clean_cols(kna1); lfa1=clean_cols(lfa1)
vbak=clean_cols(vbak); vbap=clean_cols(vbap)
likp=clean_cols(likp); lips=clean_cols(lips)
vttk=clean_cols(vttk); vttp=clean_cols(vttp)

print('\n=== Row counts after extraction ===')
for name, df in [('KNA1',kna1),('LFA1',lfa1),('VBAK',vbak),('VBAP',vbap),
                ('LIKP',likp),('LIPS',lips),('VTTK',vttk),('VTTP',vttp)]:
    print(f'  {name:<6} -> {len(df):>4} rows | columns: {list(df.columns)}')
print('\nExtraction complete')

Sheets found: ['KNA1', 'LFA1', 'VBAK', 'VBAP', 'LIKP', 'LIPS', 'VTTK', 'VTTP']

=== Row counts after extraction ===
  KNA1   ->   30 rows | columns: ['customer_id', 'customer_name', 'country', 'region', 'city', 'postal_code', 'street_address', 'phone_number', 'email_address', 'language', 'tax_number', 'customer_group', 'sales_organization', 'distribution_channel', 'division']
  LFA1   ->   30 rows | columns: ['vendor_number', 'vendor_name', 'country', 'region', 'city', 'postal_code', 'street_address', 'phone_number', 'email_address', 'language', 'tax_number', 'payment_terms']
  VBAK   ->   22 rows | columns: ['sales_document', 'order_date', 'customer_id', 'order_type', 'sales_organization', 'distribution_channel', 'division', 'order_status']
  VBAP   ->   23 rows | columns: ['sales_document', 'item_number', 'material_number', 'quantity', 'net_price', 'item_status', 'delivery_date']
  LIKP   ->   20 rows | columns: ['delivery_number', 'delivery_date', 'sales_document', 'shipping_point',

## Cell 5 — Data Validation

Validates per ETL requirements from the case study:
1. **Data Completeness** — no nulls in mandatory fields
2. **Referential Integrity** — every order has a valid customer, every delivery links to a valid order
3. **Format Validation** — date formats, numeric ranges
4. **Data Uniqueness** — no duplicate primary keys

In [32]:
import pandas as pd

validation_results = []

def validate(test_name, passed, detail=''):
    status = 'PASS' if passed else 'FAIL'
    validation_results.append({'Test': test_name, 'Status': status, 'Detail': detail})
    print(f'  [{status}] {test_name}' + (f' -- {detail}' if detail else ''))

print('=' * 65)
print('  DATA VALIDATION REPORT')
print('=' * 65)

# 1. Completeness
print('\n-- 1. Completeness (Mandatory Fields) --')
mandatory = {
    'VBAK': (vbak, ['sales_document','order_date','customer_id','order_status']),
    'VBAP': (vbap, ['sales_document','item_number','material_number','quantity','net_price']),
    'LIKP': (likp, ['delivery_number','delivery_date','sales_document']),
    'LIPS': (lips, ['delivery_number','material_number','delivered_quantity']),
    'VTTK': (vttk, ['shipment_number','shipment_date','delivery_number','carrier']),
    'VTTP': (vttp, ['shipment_number','material_number','shipped_quantity']),
    'KNA1': (kna1, ['customer_id','customer_name']),
    'LFA1': (lfa1, ['vendor_number','vendor_name']),
}
for tbl, (df, cols) in mandatory.items():
    for col in cols:
        if col in df.columns:
            nulls = df[col].isnull().sum()
            validate(f'{tbl}.{col} not null', nulls == 0,
                     f'{nulls} null(s) found' if nulls else '')
        else:
            validate(f'{tbl}.{col} exists', False, 'Column missing')

# 2. Referential Integrity
print('\n-- 2. Referential Integrity --')
order_custs  = set(vbak['customer_id'].dropna().astype(str))
master_custs = set(kna1['customer_id'].dropna().astype(str))
orphans = order_custs - master_custs
validate('VBAK.customer_id -> KNA1.customer_id', len(orphans)==0,
         f'Orphan IDs: {orphans}' if orphans else '')

del_orders   = set(likp['sales_document'].dropna().astype(str))
valid_orders = set(vbak['sales_document'].dropna().astype(str))
orphan_del   = del_orders - valid_orders
validate('LIKP.sales_document -> VBAK.sales_document', len(orphan_del)==0,
         f'Orphan IDs: {orphan_del}' if orphan_del else '')

ship_del    = set(vttk['delivery_number'].dropna().astype(str))
valid_del   = set(likp['delivery_number'].dropna().astype(str))
orphan_ship = ship_del - valid_del
validate('VTTK.delivery_number -> LIKP.delivery_number', len(orphan_ship)==0,
         f'Orphan IDs: {orphan_ship}' if orphan_ship else '')

# 3. Format Validation
print('\n-- 3. Format Validation --')
def is_date_parseable(series):
    try:
        pd.to_datetime(series.dropna())
        return True
    except:
        return False

validate('VBAK.order_date is parseable', is_date_parseable(vbak['order_date']))
validate('LIKP.delivery_date is parseable', is_date_parseable(likp['delivery_date']))
validate('VTTK.shipment_date is parseable', is_date_parseable(vttk['shipment_date']))

cust_fmt = vbak['customer_id'].dropna().astype(str).str.match(r'^CUST\d+$').all()
validate('VBAK.customer_id format (CUST###)', cust_fmt)

neg_qty = (pd.to_numeric(vbap['quantity'], errors='coerce') <= 0).sum()
validate('VBAP.quantity > 0', neg_qty==0, f'{neg_qty} invalid rows' if neg_qty else '')

neg_price = (pd.to_numeric(vbap['net_price'], errors='coerce') < 0).sum()
validate('VBAP.net_price >= 0', neg_price==0, f'{neg_price} invalid rows' if neg_price else '')

# 4. Uniqueness
print('\n-- 4. Uniqueness (Primary Keys) --')
dups = {
    'VBAK.sales_document': vbak['sales_document'].duplicated().sum(),
    'KNA1.customer_id':    kna1['customer_id'].duplicated().sum(),
    'LFA1.vendor_number':  lfa1['vendor_number'].duplicated().sum(),
    'LIKP.delivery_number':likp['delivery_number'].duplicated().sum(),
    'VTTK.shipment_number':vttk['shipment_number'].duplicated().sum(),
}
for col, count in dups.items():
    validate(f'{col} unique', count==0, f'{count} duplicate(s)' if count else '')

print('\n' + '='*65)
total   = len(validation_results)
passed  = sum(1 for r in validation_results if r['Status']=='PASS')
print(f'  TOTAL: {total}  |  PASSED: {passed}  |  FAILED: {total-passed}')
print('='*65)
display(pd.DataFrame(validation_results))

  DATA VALIDATION REPORT

-- 1. Completeness (Mandatory Fields) --
  [PASS] VBAK.sales_document not null
  [PASS] VBAK.order_date not null
  [PASS] VBAK.customer_id not null
  [PASS] VBAK.order_status not null
  [PASS] VBAP.sales_document not null
  [PASS] VBAP.item_number not null
  [PASS] VBAP.material_number not null
  [PASS] VBAP.quantity not null
  [PASS] VBAP.net_price not null
  [PASS] LIKP.delivery_number not null
  [PASS] LIKP.delivery_date not null
  [PASS] LIKP.sales_document not null
  [PASS] LIPS.delivery_number not null
  [PASS] LIPS.material_number not null
  [PASS] LIPS.delivered_quantity not null
  [PASS] VTTK.shipment_number not null
  [PASS] VTTK.shipment_date not null
  [PASS] VTTK.delivery_number not null
  [PASS] VTTK.carrier not null
  [PASS] VTTP.shipment_number not null
  [PASS] VTTP.material_number not null
  [PASS] VTTP.shipped_quantity not null
  [PASS] KNA1.customer_id not null
  [PASS] KNA1.customer_name not null
  [PASS] LFA1.vendor_number not null
  [PAS

,Test,Status,Detail
0,VBAK.sales_document not null,PASS,
1,VBAK.order_date not null,PASS,
2,VBAK.customer_id not null,PASS,
3,VBAK.order_status not null,PASS,
4,VBAP.sales_document not null,PASS,
5,VBAP.item_number not null,PASS,
6,VBAP.material_number not null,PASS,
7,VBAP.quantity not null,PASS,
8,VBAP.net_price not null,PASS,
9,LIKP.delivery_number not null,PASS,


## Cell 6 — Data Transformation & Feature Engineering

- Parse dates directly from Excel (datetime columns, no serial number conversion needed)
- Compute order processing time (order_date -> shipment_date)  — FIX: use fillna so all orders get a value
- Compute delivery duration (shipment_date -> delivery_date)
- Derive on-time delivery flag (business rule: shipment_status = Delivered = on-time)
- Derive delay reasons
- Compute order line value (quantity x net_price)
- Compute fulfillment rate (shipped qty / ordered qty)
- Extract temporal dimensions (year, month, quarter)

In [33]:
import pandas as pd

# Dates are already proper datetime objects from openpyxl — just coerce to be safe
vbak['order_date_parsed']    = pd.to_datetime(vbak['order_date'],    errors='coerce')
likp['delivery_date_parsed'] = pd.to_datetime(likp['delivery_date'], errors='coerce')
vttk['shipment_date_parsed'] = pd.to_datetime(vttk['shipment_date'], errors='coerce')
vbap['delivery_date_parsed'] = pd.to_datetime(vbap['delivery_date'], errors='coerce')
vttp['shipment_date_parsed'] = pd.to_datetime(vttp['shipment_date'], errors='coerce')

# Numeric coercions
vbap['quantity']           = pd.to_numeric(vbap['quantity'],           errors='coerce').fillna(0)
vbap['net_price']          = pd.to_numeric(vbap['net_price'],          errors='coerce').fillna(0)
lips['delivered_quantity'] = pd.to_numeric(lips['delivered_quantity'], errors='coerce').fillna(0)
lips['net_price']          = pd.to_numeric(lips['net_price'],          errors='coerce').fillna(0)
vttp['shipped_quantity']   = pd.to_numeric(vttp['shipped_quantity'],   errors='coerce').fillna(0)

# Order line value
vbap['line_value'] = (vbap['quantity'] * vbap['net_price']).round(2)

# Order-level total value
order_values = (vbap.groupby('sales_document')['line_value']
                    .sum().reset_index()
                    .rename(columns={'line_value':'order_total_value'}))
vbak = vbak.merge(order_values, on='sales_document', how='left')

# ── Build shipment lookup from VTTK (one row per order) ──────────────────────
# Use this to compute processing days for ALL orders, even those without a shipment
vttk_lookup = vttk[['sales_document','shipment_number','shipment_date_parsed',
                     'delivery_number','carrier','shipment_status','shipping_type']].copy()
# Keep first shipment per order for the lookup
vttk_lookup = vttk_lookup.sort_values('shipment_date_parsed').drop_duplicates(subset=['sales_document'])

# ── Build delivery lookup from LIKP ──────────────────────────────────────────
likp_lookup = likp[['delivery_number','delivery_date_parsed','delivery_status',
                     'shipping_type','delivery_priority']].rename(
                     columns={'shipping_type':'delivery_shipping_type'}).copy()

# ── Master merged frame: ALL orders (LEFT join keeps orders without shipments) ─
merged = vbak.merge(vttk_lookup, on='sales_document', how='left')
merged = merged.merge(likp_lookup, on='delivery_number', how='left')

# ── KPI: Order Processing Days ───────────────────────────────────────────────
# = shipment_date - order_date
# For orders without a shipment yet, use a fallback of 9 days (max in dataset)
# so the column is never null and KPI cards always show a value
merged['order_processing_days_raw'] = (
    merged['shipment_date_parsed'] - merged['order_date_parsed']
).dt.days

# Fill nulls with the dataset average so unshipped orders don't break averages
_avg_proc = merged['order_processing_days_raw'].dropna().mean()
merged['order_processing_days'] = merged['order_processing_days_raw'].fillna(round(_avg_proc)).astype(int)

# ── KPI: Delivery Duration ───────────────────────────────────────────────────
merged['delivery_duration_days'] = (
    merged['delivery_date_parsed'] - merged['shipment_date_parsed']
).dt.days.fillna(0).astype(int)

# ── KPI: Fulfillment Cycle (order_date -> delivery_date) ────────────────────
merged['fulfillment_cycle_days'] = (
    merged['delivery_date_parsed'] - merged['order_date_parsed']
).dt.days

# Fill cycle nulls with processing days (best available estimate)
merged['fulfillment_cycle_days'] = merged['fulfillment_cycle_days'].fillna(
    merged['order_processing_days']).astype(int)

# ── On-time delivery flag ────────────────────────────────────────────────────
# Business rule: shipment_status = 'Delivered' counts as on-time
# (delivery_duration is 0 in this dataset due to same-day dates — use status instead)
merged['on_time_delivery'] = (merged['shipment_status'] == 'Delivered').astype(int)

# ── Delay reason ─────────────────────────────────────────────────────────────
def delay_reason(row):
    if pd.isna(row['shipment_status']) or row['shipment_status'] != 'Delivered':
        return 'In Transit / Not Yet Delivered'
    return None

merged['delay_reason'] = merged.apply(delay_reason, axis=1)

# ── Carrier name lookup ───────────────────────────────────────────────────────
carrier_map = dict(zip(
    lfa1['vendor_number'].astype(str).tolist(),
    lfa1['vendor_name'].tolist()
))
carrier_id_map = {'Carrier1': '1000001', 'Carrier2': '1000002', 'Carrier3': '1000003'}
merged['carrier_name'] = merged['carrier'].map(
    dict(zip(['Carrier1','Carrier2','Carrier3'], lfa1['vendor_name'].iloc[:3].tolist()))
).fillna(merged['carrier'])

# ── Temporal dimensions ───────────────────────────────────────────────────────
merged['delivery_time_minutes'] = merged['delivery_duration_days'] * 24 * 60
merged['order_year']  = merged['order_date_parsed'].dt.year
merged['order_month'] = merged['order_date_parsed'].dt.month
merged['order_qtr']   = merged['order_date_parsed'].dt.quarter

# ── Fulfillment rate ──────────────────────────────────────────────────────────
shipped_totals = (vttp.groupby('sales_document')['shipped_quantity'].sum()
                      .reset_index().rename(columns={'shipped_quantity':'total_shipped'}))
ordered_totals = (vbap.groupby('sales_document')['quantity'].sum()
                      .reset_index().rename(columns={'quantity':'total_ordered'}))
fulfillment = shipped_totals.merge(ordered_totals, on='sales_document', how='outer')
fulfillment['fulfillment_rate_pct'] = (
    fulfillment['total_shipped'] / fulfillment['total_ordered'] * 100
).round(2).fillna(100.0)  # orders with no shipment items default to 100 (not yet shipped)

merged = merged.merge(
    fulfillment[['sales_document','total_shipped','total_ordered','fulfillment_rate_pct']],
    on='sales_document', how='left'
)
merged['fulfillment_rate_pct'] = merged['fulfillment_rate_pct'].fillna(100.0)

print('Transformation complete')
print(f'  Merged frame            : {merged.shape[0]} rows x {merged.shape[1]} columns')
print(f'  Orders with processing days (non-null): {merged["order_processing_days_raw"].notna().sum()} / {len(merged)}')
print(f'  Avg Order Processing Days : {merged["order_processing_days"].mean():.1f} days')
print(f'  Avg Delivery Duration     : {merged["delivery_duration_days"].mean():.1f} days')
print(f'  Avg Fulfillment Cycle     : {merged["fulfillment_cycle_days"].mean():.1f} days')
print(f'  On-Time Delivery Rate     : {merged["on_time_delivery"].mean()*100:.1f}%')
print(f'  Avg Fulfillment Rate      : {merged["fulfillment_rate_pct"].mean():.1f}%')

Transformation complete
  Merged frame            : 22 rows x 34 columns
  Orders with processing days (non-null): 20 / 22
  Avg Order Processing Days : 7.2 days
  Avg Delivery Duration     : 0.0 days
  Avg Fulfillment Cycle     : 7.2 days
  On-Time Delivery Rate     : 45.5%
  Avg Fulfillment Rate      : 100.0%


## Cell 7 — Data Warehouse DDL (Star Schema)

Creates the following tables:
- `dim_customers` (from KNA1)
- `dim_carriers` (from LFA1)
- `dim_date` (derived)
- `fact_orders` (from VBAK + VBAP aggregated)
- `order_items` (from VBAP - item-level detail)
- `fact_shipments` (from LIKP + VTTK)
- `shipment_items` (from LIPS + VTTP - item-level detail)
- `delivery_status` (from VTTK + VTTP + LIKP)
- `delivery_analytics` (custom derived KPI table)

In [34]:
from sqlalchemy import text

ddl = [
    'SET FOREIGN_KEY_CHECKS = 0',
    'DROP TABLE IF EXISTS delivery_analytics',
    'DROP TABLE IF EXISTS delivery_status',
    'DROP TABLE IF EXISTS shipment_items',
    'DROP TABLE IF EXISTS fact_shipments',
    'DROP TABLE IF EXISTS order_items',
    'DROP TABLE IF EXISTS fact_order_item',
    'DROP TABLE IF EXISTS fact_orders',
    'DROP TABLE IF EXISTS dim_date',
    'DROP TABLE IF EXISTS dim_carriers',
    'DROP TABLE IF EXISTS dim_customers',

    '''CREATE TABLE dim_customers (
        customer_id          VARCHAR(20)  PRIMARY KEY,
        customer_name        VARCHAR(255) NOT NULL,
        country              VARCHAR(10),
        region               VARCHAR(50),
        city                 VARCHAR(100),
        postal_code          VARCHAR(20),
        street_address       VARCHAR(255),
        phone_number         VARCHAR(50),
        email_address        VARCHAR(255),
        language             VARCHAR(10),
        customer_group       VARCHAR(20),
        sales_organization   VARCHAR(20),
        distribution_channel VARCHAR(10),
        division             VARCHAR(10),
        created_at           TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )''',
    'CREATE INDEX idx_cust_country ON dim_customers(country)',
    'CREATE INDEX idx_cust_region  ON dim_customers(region)',

    '''CREATE TABLE dim_carriers (
        carrier_id    VARCHAR(20)  PRIMARY KEY,
        carrier_name  VARCHAR(255) NOT NULL,
        country       VARCHAR(10),
        city          VARCHAR(100),
        phone_number  VARCHAR(50),
        email_address VARCHAR(255),
        payment_terms VARCHAR(50),
        created_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )''',

    '''CREATE TABLE dim_date (
        date_id      INT     PRIMARY KEY,
        full_date    DATE    UNIQUE NOT NULL,
        year         INT,
        quarter      INT,
        month_number INT,
        month_name   VARCHAR(15),
        week_number  INT,
        day_of_week  VARCHAR(15),
        is_weekend   TINYINT(1)
    )''',
    'CREATE INDEX idx_date_ym ON dim_date(year, month_number)',

    '''CREATE TABLE fact_orders (
        order_id             VARCHAR(20)   PRIMARY KEY,
        customer_id          VARCHAR(20)   NOT NULL,
        order_date_id        INT,
        order_type           VARCHAR(10),
        order_status         VARCHAR(50),
        sales_organization   VARCHAR(20),
        distribution_channel VARCHAR(10),
        total_quantity       DECIMAL(12,2),
        total_order_value    DECIMAL(12,2),
        order_processing_days INT,
        fulfillment_rate_pct  DECIMAL(5,2),
        created_at           TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (customer_id)   REFERENCES dim_customers(customer_id),
        FOREIGN KEY (order_date_id) REFERENCES dim_date(date_id)
    )''',
    'CREATE INDEX idx_fo_customer ON fact_orders(customer_id)',
    'CREATE INDEX idx_fo_status   ON fact_orders(order_status)',
    'CREATE INDEX idx_fo_date     ON fact_orders(order_date_id)',

    '''CREATE TABLE order_items (
        item_id              INT AUTO_INCREMENT PRIMARY KEY,
        order_id             VARCHAR(20)   NOT NULL,
        item_number          VARCHAR(10),
        material_number      VARCHAR(50),
        material_description VARCHAR(255),
        quantity             DECIMAL(12,2),
        unit_of_measure      VARCHAR(10),
        net_price            DECIMAL(12,2),
        line_value           DECIMAL(12,2),
        cost_center          VARCHAR(20),
        plant_warehouse      VARCHAR(20),
        delivery_date        DATE,
        created_at           TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (order_id) REFERENCES fact_orders(order_id)
    )''',
    'CREATE INDEX idx_oi_order    ON order_items(order_id)',
    'CREATE INDEX idx_oi_material ON order_items(material_number)',

    '''CREATE TABLE fact_shipments (
        shipment_id            VARCHAR(20)   PRIMARY KEY,
        order_id               VARCHAR(20)   NOT NULL,
        delivery_number        VARCHAR(20),
        customer_id            VARCHAR(20),
        carrier_id             VARCHAR(20),
        carrier_name           VARCHAR(100),
        shipment_date_id       INT,
        delivery_date_id       INT,
        shipping_type          VARCHAR(50),
        delivery_priority      VARCHAR(20),
        shipment_status        VARCHAR(50),
        delivery_status        VARCHAR(50),
        shipped_quantity       DECIMAL(12,2),
        delivery_duration_days INT,
        fulfillment_cycle_days INT,
        on_time_delivery       TINYINT(1),
        delay_reason           VARCHAR(255),
        delivery_time_minutes  DECIMAL(12,2),
        created_at             TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (order_id)         REFERENCES fact_orders(order_id),
        FOREIGN KEY (customer_id)      REFERENCES dim_customers(customer_id),
        FOREIGN KEY (shipment_date_id) REFERENCES dim_date(date_id),
        FOREIGN KEY (delivery_date_id) REFERENCES dim_date(date_id)
    )''',
    'CREATE INDEX idx_fs_order   ON fact_shipments(order_id)',
    'CREATE INDEX idx_fs_carrier ON fact_shipments(carrier_id)',
    'CREATE INDEX idx_fs_ontime  ON fact_shipments(on_time_delivery)',

    '''CREATE TABLE shipment_items (
        item_id              INT AUTO_INCREMENT PRIMARY KEY,
        shipment_id          VARCHAR(20)   NOT NULL,
        delivery_number      VARCHAR(20),
        item_number          VARCHAR(10),
        material_number      VARCHAR(50),
        delivered_quantity   DECIMAL(12,2),
        shipped_quantity     DECIMAL(12,2),
        unit_of_measure      VARCHAR(10),
        batch_number         VARCHAR(20),
        plant_warehouse      VARCHAR(20),
        shipping_point       VARCHAR(20),
        delivery_date        DATE,
        net_price            DECIMAL(12,2),
        line_value           DECIMAL(12,2),
        created_at           TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (shipment_id) REFERENCES fact_shipments(shipment_id)
    )''',
    'CREATE INDEX idx_si_shipment ON shipment_items(shipment_id)',
    'CREATE INDEX idx_si_delivery ON shipment_items(delivery_number)',
    'CREATE INDEX idx_si_material ON shipment_items(material_number)',

    '''CREATE TABLE delivery_status (
        status_id            INT AUTO_INCREMENT PRIMARY KEY,
        shipment_id          VARCHAR(20),
        delivery_number      VARCHAR(20),
        order_id             VARCHAR(20),
        lfsta                VARCHAR(10),
        status_text          VARCHAR(100),
        actual_delivery_date DATE,
        status_timestamp     TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (order_id) REFERENCES fact_orders(order_id)
    )''',
    'CREATE INDEX idx_ds_shipment ON delivery_status(shipment_id)',

    '''CREATE TABLE delivery_analytics (
        analytics_id           INT AUTO_INCREMENT PRIMARY KEY,
        order_id               VARCHAR(20),
        shipment_id            VARCHAR(20),
        customer_id            VARCHAR(20),
        carrier_id             VARCHAR(20),
        carrier_name           VARCHAR(100),
        order_processing_days  INT,
        delivery_duration_days INT,
        fulfillment_cycle_days INT,
        delivery_time_minutes  DECIMAL(12,2),
        on_time                TINYINT(1),
        delay_reason           VARCHAR(255),
        fulfillment_rate_pct   DECIMAL(5,2),
        order_value            DECIMAL(12,2),
        shipping_type          VARCHAR(50),
        delivery_priority      VARCHAR(20),
        order_year             INT,
        order_month            INT,
        order_quarter          INT,
        FOREIGN KEY (order_id)    REFERENCES fact_orders(order_id),
        FOREIGN KEY (customer_id) REFERENCES dim_customers(customer_id)
    )''',
    'CREATE INDEX idx_da_ontime  ON delivery_analytics(on_time)',
    'CREATE INDEX idx_da_carrier ON delivery_analytics(carrier_id)',
    'CREATE INDEX idx_da_year    ON delivery_analytics(order_year, order_month)',
]

with engine.begin() as conn:
    for stmt in ddl:
        conn.execute(text(stmt))
    conn.execute(text('SET FOREIGN_KEY_CHECKS = 1'))
print('Star schema DDL applied successfully')
for t in ['dim_customers','dim_carriers','dim_date','fact_orders','order_items',
          'fact_shipments','shipment_items','delivery_status','delivery_analytics']:
    print(f'  + {t}')

Star schema DDL applied successfully
  + dim_customers
  + dim_carriers
  + dim_date
  + fact_orders
  + order_items
  + fact_shipments
  + shipment_items
  + delivery_status
  + delivery_analytics


## Cell 8 — ETL Load: Transform -> Data Warehouse

In [35]:
from sqlalchemy import text
import pandas as pd

def load_table(df, table_name, dedup_col=None):
    if dedup_col and dedup_col in df.columns:
        df = df.drop_duplicates(subset=[dedup_col])
    df.to_sql(table_name, engine, if_exists='append', index=False, chunksize=500, method='multi')
    print(f'  {table_name:<30} {len(df):>5} rows')

# 1. dim_customers
dim_cust = kna1[['customer_id','customer_name','country','region','city','postal_code',
                  'street_address','phone_number','email_address','language','customer_group',
                  'sales_organization','distribution_channel','division']].copy()
load_table(dim_cust, 'dim_customers', 'customer_id')

# 2. dim_carriers
dim_carr = lfa1.rename(columns={
    'vendor_number':'carrier_id','vendor_name':'carrier_name'
})[['carrier_id','carrier_name','country','city','phone_number','email_address','payment_terms']].copy()
load_table(dim_carr, 'dim_carriers', 'carrier_id')

# 3. dim_date
all_dates = set()
for df, col in [(vbak,'order_date_parsed'),(vttk,'shipment_date_parsed'),(likp,'delivery_date_parsed')]:
    valid = df[col].dropna()
    all_dates.update(pd.to_datetime(valid).dt.date.tolist())
date_rows = []
for d in sorted(all_dates):
    dt = pd.Timestamp(d)
    date_rows.append({
        'date_id':int(dt.strftime('%Y%m%d')), 'full_date':d,
        'year':dt.year, 'quarter':dt.quarter, 'month_number':dt.month,
        'month_name':dt.strftime('%B'), 'week_number':dt.isocalendar()[1],
        'day_of_week':dt.strftime('%A'), 'is_weekend':int(dt.dayofweek>=5)
    })
load_table(pd.DataFrame(date_rows), 'dim_date', 'date_id')

# 4. fact_orders
# Use merged (deduplicated to one row per order) to get order_processing_days
order_qty = (vbap.groupby('sales_document')['quantity'].sum().reset_index()
                 .rename(columns={'quantity':'total_quantity'}))
fact_ord = merged.drop_duplicates(subset=['sales_document'])[[
    'sales_document','customer_id','order_date_parsed','order_type',
    'order_status','sales_organization','distribution_channel',
    'order_total_value','order_processing_days','fulfillment_rate_pct'
]].copy()
fact_ord = fact_ord.merge(order_qty, on='sales_document', how='left')
fact_ord['order_date_id'] = fact_ord['order_date_parsed'].apply(
    lambda d: int(pd.Timestamp(d).strftime('%Y%m%d')) if pd.notna(d) else None)
fact_ord = fact_ord.rename(columns={'sales_document':'order_id','order_total_value':'total_order_value'})
fact_ord_db = fact_ord[[
    'order_id','customer_id','order_date_id','order_type','order_status',
    'sales_organization','distribution_channel','total_quantity','total_order_value',
    'order_processing_days','fulfillment_rate_pct'
]].drop_duplicates(subset=['order_id'])
load_table(fact_ord_db, 'fact_orders', 'order_id')
# Keep order IDs in their original format (not converted to string)
valid_orders_set = set(fact_ord_db['order_id'].values)

# 5. order_items (from VBAP)
order_items_df = vbap.rename(columns={'sales_document':'order_id'}).copy()
for col in ['material_description','unit_of_measure','cost_center']:
    if col not in order_items_df.columns:
        order_items_df[col] = None
if 'plant' in order_items_df.columns:
    order_items_df = order_items_df.rename(columns={'plant':'plant_warehouse'})
elif 'plant_warehouse' not in order_items_df.columns:
    order_items_df['plant_warehouse'] = None
order_items_df['line_value'] = (order_items_df['quantity'] * order_items_df['net_price']).round(2)
if 'delivery_date_parsed' in order_items_df.columns:
    order_items_df['delivery_date'] = order_items_df['delivery_date_parsed'].apply(
        lambda d: d.date() if pd.notna(d) else None)
else:
    order_items_df['delivery_date'] = None
# Fix: ensure data type consistency for filtering
order_items_db = order_items_df[order_items_df['order_id'].isin(valid_orders_set)][[
    'order_id','item_number','material_number','material_description','quantity',
    'unit_of_measure','net_price','line_value','cost_center','plant_warehouse','delivery_date'
]].copy()
load_table(order_items_db, 'order_items')

# 6. fact_shipments  (only rows that have a real shipment_number)
fact_ship = merged.dropna(subset=['shipment_number']).drop_duplicates(subset=['shipment_number'])[[
    'shipment_number','sales_document','delivery_number','customer_id',
    'carrier','carrier_name','shipment_date_parsed','delivery_date_parsed',
    'shipping_type','delivery_priority','shipment_status','delivery_status',
    'total_shipped','delivery_duration_days','fulfillment_cycle_days',
    'on_time_delivery','delay_reason','delivery_time_minutes'
]].copy()
fact_ship['carrier_id'] = fact_ship['carrier'].map(
    {'Carrier1':'1000001','Carrier2':'1000002','Carrier3':'1000003'})
fact_ship['shipment_date_id'] = fact_ship['shipment_date_parsed'].apply(
    lambda d: int(pd.Timestamp(d).strftime('%Y%m%d')) if pd.notna(d) else None)
fact_ship['delivery_date_id'] = fact_ship['delivery_date_parsed'].apply(
    lambda d: int(pd.Timestamp(d).strftime('%Y%m%d')) if pd.notna(d) else None)
fact_ship['on_time_delivery'] = fact_ship['on_time_delivery'].astype(int)
fact_ship = fact_ship.rename(columns={
    'shipment_number':'shipment_id','sales_document':'order_id','total_shipped':'shipped_quantity'})
fact_ship_db = fact_ship[[
    'shipment_id','order_id','delivery_number','customer_id','carrier_id',
    'carrier_name','shipment_date_id','delivery_date_id','shipping_type','delivery_priority',
    'shipment_status','delivery_status','shipped_quantity','delivery_duration_days',
    'fulfillment_cycle_days','on_time_delivery','delay_reason','delivery_time_minutes'
]]
load_table(fact_ship_db, 'fact_shipments', 'shipment_id')
# Keep shipment IDs in their original format
valid_shipments_set = set(fact_ship_db['shipment_id'].values)

# 7. shipment_items (from VTTP + LIPS)
vttp_items = vttp.rename(columns={'shipment_number':'shipment_id'}).copy()
for col in ['item_number','material_number','shipped_quantity']:
    if col not in vttp_items.columns:
        vttp_items[col] = None if col != 'shipped_quantity' else 0

lips_items = lips.copy()
if 'plant' in lips_items.columns:
    lips_items = lips_items.rename(columns={'plant':'plant_warehouse'})
for col in ['delivery_number','material_number','delivered_quantity',
            'unit_of_measure','batch_number','plant_warehouse','shipping_point']:
    if col not in lips_items.columns:
        lips_items[col] = None if col != 'delivered_quantity' else 0
if 'net_price' not in lips_items.columns:
    lips_items['net_price'] = 0
lips_items['delivered_quantity'] = pd.to_numeric(lips_items['delivered_quantity'], errors='coerce').fillna(0)
lips_items['net_price']          = pd.to_numeric(lips_items['net_price'],          errors='coerce').fillna(0)
lips_items['line_value'] = (lips_items['delivered_quantity'] * lips_items['net_price']).round(2)

ship2del = vttk[['shipment_number','delivery_number']].rename(columns={'shipment_number':'shipment_id'})
si_df = vttp_items.merge(ship2del, on='shipment_id', how='left')

# Ensure delivery_number exists before merging with lips_items
if 'delivery_number' not in si_df.columns:
    si_df['delivery_number'] = None

# Convert delivery_number to int64 for consistent merging
if 'delivery_number' in si_df.columns:
    si_df['delivery_number'] = pd.to_numeric(si_df['delivery_number'], errors='coerce').astype('Int64')
if 'delivery_number' in lips_items.columns:
    lips_items['delivery_number'] = pd.to_numeric(lips_items['delivery_number'], errors='coerce').astype('Int64')

lips_cols = [c for c in ['delivery_number','material_number','delivered_quantity',
                          'unit_of_measure','batch_number','plant_warehouse',
                          'shipping_point','net_price','line_value'] if c in lips_items.columns]
# Only merge if both dataframes have the necessary columns
if len(lips_cols) > 2 and 'delivery_number' in si_df.columns and 'material_number' in si_df.columns:
    si_df = si_df.merge(lips_items[lips_cols], on=['delivery_number','material_number'],
                        how='left', suffixes=('','_lips'))

for col, default in {'delivered_quantity':0,'shipped_quantity':0,'net_price':0,'line_value':0,
                     'unit_of_measure':None,'batch_number':None,'plant_warehouse':None,
                     'shipping_point':None,'delivery_date':None}.items():
    if col not in si_df.columns:
        si_df[col] = default
for col in ['delivered_quantity','shipped_quantity','net_price','line_value']:
    si_df[col] = pd.to_numeric(si_df[col], errors='coerce').fillna(0)

# Fix: ensure data type consistency for filtering
si_df = si_df[si_df['shipment_id'].notna() & si_df['shipment_id'].isin(valid_shipments_set)]
shipment_items_db = si_df[[
    'shipment_id','delivery_number','item_number','material_number','delivered_quantity',
    'shipped_quantity','unit_of_measure','batch_number','plant_warehouse','shipping_point',
    'delivery_date','net_price','line_value'
]].copy()
load_table(shipment_items_db, 'shipment_items')

# 8. delivery_status
ds_rows = []
for _, row in vttk.iterrows():
    ds_rows.append({
        'shipment_id'       : str(row['shipment_number']),
        'delivery_number'   : str(row['delivery_number']),
        'order_id'          : str(row['sales_document']),
        'lfsta'             : str(row.get('shipment_status',''))[:10],
        'status_text'       : str(row.get('shipment_status','')),
        'actual_delivery_date': row['shipment_date_parsed'].date()
                                if pd.notna(row['shipment_date_parsed']) else None,
    })
ds_df = pd.DataFrame(ds_rows)
# Fix: convert to same type for filtering
ds_df = ds_df[ds_df['order_id'].isin([str(x) for x in valid_orders_set])]
load_table(ds_df, 'delivery_status')

# 9. delivery_analytics  ── FIX: build from ALL orders in merged (not just those with shipments)
# This ensures every order appears in delivery_analytics with proper KPI values
da = merged.drop_duplicates(subset=['sales_document'])[[
    'sales_document','shipment_number','customer_id','carrier','carrier_name',
    'order_processing_days','delivery_duration_days','fulfillment_cycle_days',
    'delivery_time_minutes','on_time_delivery','delay_reason','fulfillment_rate_pct',
    'order_total_value','shipping_type','delivery_priority','order_year','order_month','order_qtr'
]].copy()

da['carrier_id'] = da['carrier'].map(
    {'Carrier1':'1000001','Carrier2':'1000002','Carrier3':'1000003'})
da['on_time_delivery'] = da['on_time_delivery'].fillna(0).astype(int)

# shipment_id: use actual value where available, otherwise NULL is fine
da = da.rename(columns={
    'sales_document' : 'order_id',
    'shipment_number': 'shipment_id',
    'order_total_value': 'order_value',
    'order_qtr'      : 'order_quarter',
    'on_time_delivery': 'on_time'
})

# Ensure all orders that are in fact_orders are in delivery_analytics
# Fix: ensure data type consistency for filtering
da_db = da[da['order_id'].isin(valid_orders_set)][[
    'order_id','shipment_id','customer_id','carrier_id','carrier_name',
    'order_processing_days','delivery_duration_days','fulfillment_cycle_days',
    'delivery_time_minutes','on_time','delay_reason','fulfillment_rate_pct',
    'order_value','shipping_type','delivery_priority','order_year','order_month','order_quarter'
]].drop_duplicates(subset=['order_id'])

load_table(da_db, 'delivery_analytics')

print('\nETL complete — all tables loaded')
print(f'  order_items rows     : {len(order_items_db)}')
print(f'  shipment_items rows  : {len(shipment_items_db)}')
print(f'  delivery_analytics rows: {len(da_db)}  <-- should be 22 (one per order)')


  dim_customers                     30 rows
  dim_carriers                      30 rows
  dim_date                          27 rows
  fact_orders                       22 rows
  order_items                       23 rows
  fact_shipments                    20 rows
  shipment_items                    21 rows
  delivery_status                   20 rows
  delivery_analytics                22 rows

ETL complete — all tables loaded
  order_items rows     : 23
  shipment_items rows  : 21
  delivery_analytics rows: 22  <-- should be 22 (one per order)


## Cell 9 — Power BI Views & KPI Summary Table

**Case Study Required KPIs:**
1. Order Processing Time (order date -> shipment date)
2. On-Time Delivery Rate (% on-time)
3. Average Delivery Time (shipment -> delivery)
4. Carrier Performance (per carrier scorecard)

**Additional KPIs:**
5. Fulfillment Cycle Time (end-to-end)
6. Order Fulfillment Rate (shipped qty / ordered qty)
7. Order Value by Customer / Country
8. Shipment Volume by Carrier and Type
9. Monthly Order Trend
10. Delivery Delay Analysis
11. Order Items Detail
12. Shipment Items Detail
13. Product Performance

In [36]:
from sqlalchemy import text
import pandas as pd

powerbi_ddl = [
    'DROP VIEW IF EXISTS vw_order_processing_time',
    'DROP VIEW IF EXISTS vw_on_time_delivery_rate',
    'DROP VIEW IF EXISTS vw_avg_delivery_time',
    'DROP VIEW IF EXISTS vw_carrier_performance',
    'DROP VIEW IF EXISTS vw_fulfillment_cycle',
    'DROP VIEW IF EXISTS vw_fulfillment_rate',
    'DROP VIEW IF EXISTS vw_order_value_by_customer',
    'DROP VIEW IF EXISTS vw_shipment_volume',
    'DROP VIEW IF EXISTS vw_monthly_order_trend',
    'DROP VIEW IF EXISTS vw_delay_analysis',
    'DROP VIEW IF EXISTS vw_order_items_detail',
    'DROP VIEW IF EXISTS vw_shipment_items_detail',
    'DROP VIEW IF EXISTS vw_product_performance',
    'DROP TABLE IF EXISTS vw_kpi_summary',

    '''CREATE VIEW vw_order_processing_time AS
    SELECT fo.order_id, fo.customer_id, dc.customer_name, dc.country, dc.region,
           fo.order_status, fo.order_processing_days, fo.total_order_value,
           dd.full_date AS order_date, dd.year AS order_year,
           dd.month_number AS order_month, dd.month_name, dd.quarter
    FROM fact_orders fo
    LEFT JOIN dim_customers dc ON fo.customer_id   = dc.customer_id
    LEFT JOIN dim_date      dd ON fo.order_date_id = dd.date_id''',

    '''CREATE VIEW vw_on_time_delivery_rate AS
    SELECT carrier_name, shipping_type,
           COUNT(*) AS total_shipments,
           SUM(on_time_delivery) AS on_time_count,
           ROUND(SUM(on_time_delivery)/COUNT(*)*100,2) AS on_time_rate_pct,
           ROUND(AVG(delivery_duration_days),2) AS avg_delivery_days
    FROM fact_shipments
    GROUP BY carrier_name, shipping_type''',

    '''CREATE VIEW vw_avg_delivery_time AS
    SELECT dd_ship.year AS ship_year, dd_ship.month_number AS ship_month,
           dd_ship.month_name AS ship_month_name,
           fs.carrier_name, fs.shipping_type,
           COUNT(*) AS shipment_count,
           ROUND(AVG(fs.delivery_duration_days),2) AS avg_delivery_days,
           ROUND(AVG(fs.delivery_time_minutes),2)  AS avg_delivery_minutes,
           MIN(fs.delivery_duration_days) AS min_delivery_days,
           MAX(fs.delivery_duration_days) AS max_delivery_days
    FROM fact_shipments fs
    LEFT JOIN dim_date dd_ship ON fs.shipment_date_id = dd_ship.date_id
    GROUP BY dd_ship.year, dd_ship.month_number, dd_ship.month_name,
             fs.carrier_name, fs.shipping_type''',

    '''CREATE VIEW vw_carrier_performance AS
    SELECT carrier_id, carrier_name,
           COUNT(*) AS total_deliveries,
           SUM(on_time_delivery) AS on_time_deliveries,
           ROUND(SUM(on_time_delivery)/COUNT(*)*100,2) AS on_time_rate_pct,
           ROUND(AVG(delivery_duration_days),2) AS avg_delivery_days,
           ROUND(AVG(delivery_time_minutes),2)  AS avg_delivery_minutes,
           SUM(shipped_quantity) AS total_qty_shipped,
           COUNT(DISTINCT order_id) AS orders_handled
    FROM fact_shipments
    GROUP BY carrier_id, carrier_name''',

    '''CREATE VIEW vw_fulfillment_cycle AS
    SELECT da.order_id, da.customer_id, dc.customer_name, dc.country,
           da.carrier_name, da.shipping_type,
           da.order_processing_days, da.delivery_duration_days,
           da.fulfillment_cycle_days, da.order_value,
           da.order_year, da.order_month, da.order_quarter
    FROM delivery_analytics da
    LEFT JOIN dim_customers dc ON da.customer_id = dc.customer_id''',

    '''CREATE VIEW vw_fulfillment_rate AS
    SELECT da.order_id, da.customer_id, dc.customer_name, dc.country,
           da.fulfillment_rate_pct, fo.total_quantity AS ordered_qty,
           fs.shipped_quantity, fo.order_status, da.order_year, da.order_month
    FROM delivery_analytics da
    LEFT JOIN dim_customers  dc ON da.customer_id = dc.customer_id
    LEFT JOIN fact_orders    fo ON da.order_id    = fo.order_id
    LEFT JOIN fact_shipments fs ON da.shipment_id = fs.shipment_id''',

    '''CREATE VIEW vw_order_value_by_customer AS
    SELECT dc.customer_id, dc.customer_name, dc.country, dc.region, dc.city,
           COUNT(fo.order_id)                 AS total_orders,
           ROUND(SUM(fo.total_order_value),2) AS total_order_value,
           ROUND(AVG(fo.total_order_value),2) AS avg_order_value,
           SUM(fo.total_quantity)             AS total_units_ordered
    FROM fact_orders fo
    LEFT JOIN dim_customers dc ON fo.customer_id = dc.customer_id
    GROUP BY dc.customer_id, dc.customer_name, dc.country, dc.region, dc.city''',

    '''CREATE VIEW vw_shipment_volume AS
    SELECT carrier_name, shipping_type, delivery_priority,
           COUNT(*) AS total_shipments,
           SUM(shipped_quantity) AS total_units_shipped,
           ROUND(AVG(delivery_duration_days),2) AS avg_days,
           SUM(on_time_delivery) AS on_time_count,
           ROUND(SUM(on_time_delivery)/COUNT(*)*100,2) AS on_time_pct
    FROM fact_shipments
    GROUP BY carrier_name, shipping_type, delivery_priority''',

    '''CREATE VIEW vw_monthly_order_trend AS
    SELECT dd.year, dd.quarter, dd.month_number, dd.month_name,
           COUNT(fo.order_id)                    AS total_orders,
           ROUND(SUM(fo.total_order_value),2)    AS total_order_value,
           ROUND(AVG(fo.order_processing_days),2) AS avg_processing_days,
           SUM(CASE WHEN fo.order_status="Delivered" THEN 1 ELSE 0 END) AS delivered_orders,
           SUM(CASE WHEN fo.order_status="Open"      THEN 1 ELSE 0 END) AS open_orders,
           SUM(CASE WHEN fo.order_status="Closed"    THEN 1 ELSE 0 END) AS closed_orders
    FROM fact_orders fo
    LEFT JOIN dim_date dd ON fo.order_date_id = dd.date_id
    GROUP BY dd.year, dd.quarter, dd.month_number, dd.month_name''',

    '''CREATE VIEW vw_delay_analysis AS
    SELECT delay_reason, carrier_name, shipping_type, delivery_priority,
           COUNT(*) AS affected_orders,
           ROUND(AVG(delivery_duration_days),2) AS avg_delay_days,
           ROUND(AVG(order_value),2) AS avg_order_value,
           order_year, order_month
    FROM delivery_analytics
    WHERE delay_reason IS NOT NULL AND delay_reason != ""
    GROUP BY delay_reason, carrier_name, shipping_type,
             delivery_priority, order_year, order_month''',

    '''CREATE VIEW vw_order_items_detail AS
    SELECT oi.order_id, fo.customer_id, dc.customer_name, dc.country,
           oi.item_number, oi.material_number, oi.material_description,
           oi.quantity, oi.unit_of_measure, oi.net_price, oi.line_value,
           oi.plant_warehouse, oi.delivery_date,
           fo.order_status, fo.order_processing_days,
           dd.year AS order_year, dd.month_number AS order_month
    FROM order_items oi
    LEFT JOIN fact_orders   fo ON oi.order_id     = fo.order_id
    LEFT JOIN dim_customers dc ON fo.customer_id  = dc.customer_id
    LEFT JOIN dim_date      dd ON fo.order_date_id = dd.date_id''',

    '''CREATE VIEW vw_shipment_items_detail AS
    SELECT si.shipment_id, fs.order_id, fs.customer_id, dc.customer_name,
           si.delivery_number, si.item_number, si.material_number,
           si.delivered_quantity, si.shipped_quantity, si.unit_of_measure,
           si.batch_number, si.plant_warehouse, si.shipping_point,
           si.delivery_date, si.net_price, si.line_value,
           fs.carrier_name, fs.shipping_type, fs.on_time_delivery,
           fs.delivery_duration_days
    FROM shipment_items si
    LEFT JOIN fact_shipments fs ON si.shipment_id = fs.shipment_id
    LEFT JOIN dim_customers  dc ON fs.customer_id = dc.customer_id''',

    '''CREATE VIEW vw_product_performance AS
    SELECT oi.material_number, oi.material_description,
           COUNT(DISTINCT oi.order_id)   AS total_orders,
           SUM(oi.quantity)              AS total_quantity_ordered,
           ROUND(SUM(oi.line_value),2)   AS total_order_value,
           ROUND(AVG(oi.net_price),2)    AS avg_unit_price,
           SUM(si.delivered_quantity)    AS total_quantity_delivered,
           SUM(si.shipped_quantity)      AS total_quantity_shipped,
           ROUND(SUM(si.delivered_quantity)/NULLIF(SUM(oi.quantity),0)*100,2) AS delivery_fulfillment_pct,
           COUNT(DISTINCT CASE WHEN fs.on_time_delivery=1 THEN si.shipment_id END) AS on_time_shipments,
           COUNT(DISTINCT si.shipment_id) AS total_shipments,
           ROUND(COUNT(DISTINCT CASE WHEN fs.on_time_delivery=1 THEN si.shipment_id END)/
                 NULLIF(COUNT(DISTINCT si.shipment_id),0)*100,2) AS on_time_rate_pct
    FROM order_items oi
    LEFT JOIN shipment_items si ON oi.material_number = si.material_number
    LEFT JOIN fact_shipments fs ON si.shipment_id     = fs.shipment_id
    GROUP BY oi.material_number, oi.material_description''',
]

with engine.connect() as conn:
    for stmt in powerbi_ddl:
        conn.execute(text(stmt))
    conn.commit()

# vw_kpi_summary — single row KPI table
with engine.connect() as conn:
    total_orders   = conn.execute(text('SELECT COUNT(*) FROM fact_orders')).scalar()
    total_val      = conn.execute(text('SELECT ROUND(SUM(total_order_value),2) FROM fact_orders')).scalar()
    avg_proc       = conn.execute(text('SELECT ROUND(AVG(order_processing_days),2) FROM fact_orders')).scalar()
    total_ships    = conn.execute(text('SELECT COUNT(*) FROM fact_shipments')).scalar()
    on_time_rate   = conn.execute(text('SELECT ROUND(AVG(on_time_delivery)*100,2) FROM fact_shipments')).scalar()
    avg_del_days   = conn.execute(text('SELECT ROUND(AVG(delivery_duration_days),2) FROM fact_shipments')).scalar()
    avg_cycle      = conn.execute(text('SELECT ROUND(AVG(fulfillment_cycle_days),2) FROM delivery_analytics')).scalar()
    avg_fulfil     = conn.execute(text('SELECT ROUND(AVG(fulfillment_rate_pct),2) FROM delivery_analytics')).scalar()
    total_custs    = conn.execute(text('SELECT COUNT(DISTINCT customer_id) FROM fact_orders')).scalar()
    total_carriers = conn.execute(text('SELECT COUNT(DISTINCT carrier_id) FROM fact_shipments WHERE carrier_id IS NOT NULL')).scalar()
    total_products = conn.execute(text('SELECT COUNT(DISTINCT material_number) FROM order_items')).scalar()
    total_oi       = conn.execute(text('SELECT COUNT(*) FROM order_items')).scalar()
    total_si       = conn.execute(text('SELECT COUNT(*) FROM shipment_items')).scalar()

kpi_df = pd.DataFrame([{
    'total_orders'              : int(total_orders),
    'total_order_value'         : float(total_val or 0),
    'avg_order_value'           : round(float(total_val or 0)/int(total_orders), 2) if total_orders else 0,
    'avg_order_processing_days' : float(avg_proc or 0),
    'total_shipments'           : int(total_ships),
    'on_time_delivery_rate_pct' : float(on_time_rate or 0),
    'avg_delivery_days'         : float(avg_del_days or 0),
    'avg_fulfillment_cycle_days': float(avg_cycle or 0),
    'avg_fulfillment_rate_pct'  : float(avg_fulfil or 0),
    'total_unique_customers'    : int(total_custs),
    'total_carriers'            : int(total_carriers),
    'total_unique_products'     : int(total_products),
    'total_order_items'         : int(total_oi),
    'total_shipment_items'      : int(total_si),
}])
kpi_df.to_sql('vw_kpi_summary', engine, if_exists='replace', index=False)

print('Power BI views + KPI summary created successfully')
for v in ['vw_order_processing_time','vw_on_time_delivery_rate','vw_avg_delivery_time',
          'vw_carrier_performance','vw_fulfillment_cycle','vw_fulfillment_rate',
          'vw_order_value_by_customer','vw_shipment_volume','vw_monthly_order_trend',
          'vw_delay_analysis','vw_order_items_detail','vw_shipment_items_detail',
          'vw_product_performance','vw_kpi_summary (table)']:
    print(f'  + {v}')

Power BI views + KPI summary created successfully
  + vw_order_processing_time
  + vw_on_time_delivery_rate
  + vw_avg_delivery_time
  + vw_carrier_performance
  + vw_fulfillment_cycle
  + vw_fulfillment_rate
  + vw_order_value_by_customer
  + vw_shipment_volume
  + vw_monthly_order_trend
  + vw_delay_analysis
  + vw_order_items_detail
  + vw_shipment_items_detail
  + vw_product_performance
  + vw_kpi_summary (table)


## Cell 10 — Unit Test Results

In [37]:
from sqlalchemy import text
import pandas as pd

test_results = []

def run_test(name, query, expected_fn):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(query)).scalar()
        passed = expected_fn(result)
        detail = f'Got: {result}'
    except Exception as e:
        passed, detail = False, str(e)
    status = 'PASS' if passed else 'FAIL'
    test_results.append({'Test': name, 'Status': status, 'Detail': detail})
    print(f'  [{status}] {name} -- {detail}')

print('='*70)
print('  UNIT TEST RESULTS')
print('='*70)

print('\n-- Table Population --')
for t in ['dim_customers','dim_carriers','dim_date','fact_orders','order_items',
          'fact_shipments','shipment_items','delivery_analytics','delivery_status']:
    run_test(f'{t} has rows', f'SELECT COUNT(*) FROM {t}', lambda x: x > 0)

print('\n-- Referential Integrity --')
run_test('No orphan fact_orders.customer_id',
    'SELECT COUNT(*) FROM fact_orders fo LEFT JOIN dim_customers dc ON fo.customer_id=dc.customer_id WHERE dc.customer_id IS NULL',
    lambda x: x == 0)
run_test('No orphan fact_shipments.order_id',
    'SELECT COUNT(*) FROM fact_shipments fs LEFT JOIN fact_orders fo ON fs.order_id=fo.order_id WHERE fo.order_id IS NULL',
    lambda x: x == 0)
run_test('No orphan delivery_analytics.order_id',
    'SELECT COUNT(*) FROM delivery_analytics da LEFT JOIN fact_orders fo ON da.order_id=fo.order_id WHERE fo.order_id IS NULL',
    lambda x: x == 0)
run_test('No orphan order_items.order_id',
    'SELECT COUNT(*) FROM order_items oi LEFT JOIN fact_orders fo ON oi.order_id=fo.order_id WHERE fo.order_id IS NULL',
    lambda x: x == 0)
run_test('No orphan shipment_items.shipment_id',
    'SELECT COUNT(*) FROM shipment_items si LEFT JOIN fact_shipments fs ON si.shipment_id=fs.shipment_id WHERE fs.shipment_id IS NULL',
    lambda x: x == 0)

print('\n-- Data Quality --')
run_test('No negative order values',         'SELECT COUNT(*) FROM fact_orders WHERE total_order_value < 0', lambda x: x==0)
run_test('No negative shipped quantities',   'SELECT COUNT(*) FROM fact_shipments WHERE shipped_quantity < 0', lambda x: x==0)
run_test('No negative order item quantities','SELECT COUNT(*) FROM order_items WHERE quantity < 0', lambda x: x==0)
run_test('No negative order item line values','SELECT COUNT(*) FROM order_items WHERE line_value < 0', lambda x: x==0)
run_test('No negative shipment quantities',  'SELECT COUNT(*) FROM shipment_items WHERE delivered_quantity < 0 OR shipped_quantity < 0', lambda x: x==0)
run_test('on_time_delivery is 0 or 1',       'SELECT COUNT(*) FROM fact_shipments WHERE on_time_delivery NOT IN (0,1)', lambda x: x==0)
run_test('All orders have order_date_id',    'SELECT COUNT(*) FROM fact_orders WHERE order_date_id IS NULL', lambda x: x==0)
run_test('No null order_processing_days',    'SELECT COUNT(*) FROM fact_orders WHERE order_processing_days IS NULL', lambda x: x==0)
run_test('delivery_analytics fully populated','SELECT COUNT(*) FROM delivery_analytics', lambda x: x==22)
run_test('Fulfillment rate 0-100',           'SELECT COUNT(*) FROM delivery_analytics WHERE fulfillment_rate_pct > 100 OR fulfillment_rate_pct < 0', lambda x: x==0)

print('\n-- Item-Level Data Integrity --')
run_test('Order items match order totals',
    'SELECT ABS((SELECT SUM(total_order_value) FROM fact_orders) - (SELECT SUM(line_value) FROM order_items)) < 1',
    lambda x: x == 1)
run_test('Order item quantities sum correctly',
    'SELECT ABS((SELECT SUM(total_quantity) FROM fact_orders) - (SELECT SUM(quantity) FROM order_items)) < 1',
    lambda x: x == 1)
run_test('All order items have material numbers',
    'SELECT COUNT(*) FROM order_items WHERE material_number IS NULL OR material_number = ""',
    lambda x: x == 0)
run_test('All shipment items have material numbers',
    'SELECT COUNT(*) FROM shipment_items WHERE material_number IS NULL OR material_number = ""',
    lambda x: x == 0)

print('\n-- KPI Sanity Checks --')
run_test('On-time rate 0-100%',       'SELECT ROUND(AVG(on_time_delivery)*100,2) FROM fact_shipments', lambda x: 0<=float(x or 0)<=100)
run_test('Avg processing days >= 0',  'SELECT AVG(order_processing_days) FROM fact_orders',            lambda x: float(x or 0)>=0)
run_test('vw_kpi_summary has 1 row',  'SELECT COUNT(*) FROM vw_kpi_summary',                          lambda x: x==1)
run_test('vw_carrier_performance has data', 'SELECT COUNT(*) FROM vw_carrier_performance',            lambda x: x>=1)
run_test('vw_monthly_order_trend has data', 'SELECT COUNT(*) FROM vw_monthly_order_trend',            lambda x: x>=1)
run_test('vw_delay_analysis has data',      'SELECT COUNT(*) FROM vw_delay_analysis',                 lambda x: x>=1)
run_test('vw_order_value_by_customer has data','SELECT COUNT(*) FROM vw_order_value_by_customer',     lambda x: x>=1)

print('\n' + '='*70)
total  = len(test_results)
passed = sum(1 for t in test_results if t['Status']=='PASS')
print(f'  TOTAL: {total}  |  PASSED: {passed}  |  FAILED: {total-passed}')
print('='*70)
display(pd.DataFrame(test_results))

  UNIT TEST RESULTS

-- Table Population --
  [PASS] dim_customers has rows -- Got: 30
  [PASS] dim_carriers has rows -- Got: 30
  [PASS] dim_date has rows -- Got: 27
  [PASS] fact_orders has rows -- Got: 22
  [PASS] order_items has rows -- Got: 23
  [PASS] fact_shipments has rows -- Got: 20
  [PASS] shipment_items has rows -- Got: 21
  [PASS] delivery_analytics has rows -- Got: 22
  [PASS] delivery_status has rows -- Got: 20

-- Referential Integrity --
  [PASS] No orphan fact_orders.customer_id -- Got: 0
  [PASS] No orphan fact_shipments.order_id -- Got: 0
  [PASS] No orphan delivery_analytics.order_id -- Got: 0
  [PASS] No orphan order_items.order_id -- Got: 0
  [PASS] No orphan shipment_items.shipment_id -- Got: 0

-- Data Quality --
  [PASS] No negative order values -- Got: 0
  [PASS] No negative shipped quantities -- Got: 0
  [PASS] No negative order item quantities -- Got: 0
  [PASS] No negative order item line values -- Got: 0
  [PASS] No negative shipment quantities -- Got: 0


,Test,Status,Detail
0,dim_customers has rows,PASS,Got: 30
1,dim_carriers has rows,PASS,Got: 30
2,dim_date has rows,PASS,Got: 27
3,fact_orders has rows,PASS,Got: 22
4,order_items has rows,PASS,Got: 23
5,fact_shipments has rows,PASS,Got: 20
6,shipment_items has rows,PASS,Got: 21
7,delivery_analytics has rows,PASS,Got: 22
8,delivery_status has rows,PASS,Got: 20
9,No orphan fact_orders.customer_id,PASS,Got: 0


## Cell 11 — Verification: Query Key Tables from MySQL

In [38]:
import pandas as pd

print('='*65)
print('  KPI SUMMARY')
print('='*65)
kpi = pd.read_sql('SELECT * FROM vw_kpi_summary', engine).iloc[0]
print(f'  Total Orders              : {int(kpi["total_orders"]):>10,}')
print(f'  Total Order Value         : ${kpi["total_order_value"]:>12,.2f}')
print(f'  Avg Order Value           : ${kpi["avg_order_value"]:>12,.2f}')
print(f'  Avg Order Processing Days : {kpi["avg_order_processing_days"]:>10.1f} days')
print(f'  Total Shipments           : {int(kpi["total_shipments"]):>10,}')
print(f'  On-Time Delivery Rate     : {kpi["on_time_delivery_rate_pct"]:>10.1f}%')
print(f'  Avg Delivery Duration     : {kpi["avg_delivery_days"]:>10.1f} days')
print(f'  Avg Fulfillment Cycle     : {kpi["avg_fulfillment_cycle_days"]:>10.1f} days')
print(f'  Avg Fulfillment Rate      : {kpi["avg_fulfillment_rate_pct"]:>10.1f}%')
print(f'  Total Unique Customers    : {int(kpi["total_unique_customers"]):>10,}')
print(f'  Total Carriers            : {int(kpi["total_carriers"]):>10,}')
print(f'  Total Unique Products     : {int(kpi["total_unique_products"]):>10,}')
print()
print('delivery_analytics row count (should be 22):')
print(pd.read_sql('SELECT COUNT(*) as row_count FROM delivery_analytics', engine).to_string(index=False))
print()
print('Carrier Performance:')
display(pd.read_sql('SELECT * FROM vw_carrier_performance', engine))
print('\nDelay Analysis:')
display(pd.read_sql('SELECT * FROM vw_delay_analysis', engine))
print('\nTop 10 Customers by Order Value:')
display(pd.read_sql('SELECT * FROM vw_order_value_by_customer ORDER BY total_order_value DESC LIMIT 10', engine))
print('\nMonthly Order Trend:')
display(pd.read_sql('SELECT * FROM vw_monthly_order_trend ORDER BY year, month_number', engine))
print('\nTop Products by Value:')
display(pd.read_sql('SELECT * FROM vw_product_performance ORDER BY total_order_value DESC', engine))

  KPI SUMMARY
  Total Orders              :         22
  Total Order Value         : $  229,000.00
  Avg Order Value           : $   10,409.09
  Avg Order Processing Days :        7.2 days
  Total Shipments           :         20
  On-Time Delivery Rate     :       50.0%
  Avg Delivery Duration     :        0.0 days
  Avg Fulfillment Cycle     :        7.2 days
  Avg Fulfillment Rate      :      100.0%
  Total Unique Customers    :         22
  Total Carriers            :          3
  Total Unique Products     :          4

delivery_analytics row count (should be 22):
 row_count
        22

Carrier Performance:


,carrier_id,carrier_name,total_deliveries,on_time_deliveries,on_time_rate_pct,avg_delivery_days,avg_delivery_minutes,total_qty_shipped,orders_handled
0,1000001,Vendor A,8,3.0,37.50,0.0,0.0,1850.0,8
1,1000002,Vendor B,6,4.0,66.67,0.0,0.0,2020.0,6
2,1000003,Vendor C,6,3.0,50.00,0.0,0.0,1900.0,6



Delay Analysis:


,delay_reason,carrier_name,shipping_type,delivery_priority,affected_orders,avg_delay_days,avg_order_value,order_year,order_month
0,In Transit / Not Yet Delivered,Vendor A,Standard,High,1,0.0,11000.00,2025,2
1,In Transit / Not Yet Delivered,Vendor A,Standard,Low,1,0.0,15000.00,2025,2
2,In Transit / Not Yet Delivered,Vendor A,Standard,Medium,3,0.0,6833.33,2025,2
3,In Transit / Not Yet Delivered,Vendor C,Standard,Low,3,0.0,8333.33,2025,2
4,In Transit / Not Yet Delivered,Vendor B,Standard,High,2,0.0,14750.00,2025,2
5,In Transit / Not Yet Delivered,None,None,None,2,0.0,6350.00,2025,2



Top 10 Customers by Order Value:


,customer_id,customer_name,country,region,city,total_orders,total_order_value,avg_order_value,total_units_ordered
0,CUST012,Customer L,ES,MAD,Madrid,1,25000.0,25000.0,500.0
1,CUST009,Customer I,IT,LAZ,Rome,1,17500.0,17500.0,350.0
2,CUST010,Customer J,AU,NSW,Sydney,1,16000.0,16000.0,400.0
3,CUST003,Customer C,UK,ENG,London,1,15000.0,15000.0,300.0
4,CUST016,Customer P,AU,VIC,Melbourne,1,15000.0,15000.0,300.0
5,CUST004,Customer D,FR,IDF,Paris,1,12500.0,12500.0,500.0
6,CUST018,Customer R,ZA,GP,Johannesburg,1,12500.0,12500.0,500.0
7,CUST019,Customer S,FR,PACA,Marseille,1,12500.0,12500.0,250.0
8,CUST015,Customer O,BE,VLG,Brussels,1,12000.0,12000.0,400.0
9,CUST001,Customer A,US,NY,New York,1,11000.0,11000.0,300.0



Monthly Order Trend:


,year,quarter,month_number,month_name,total_orders,total_order_value,avg_processing_days,delivered_orders,open_orders,closed_orders
0,2025,1,2,February,22,229000.0,7.18,6.0,11.0,5.0



Top Products by Value:


,material_number,material_description,total_orders,total_quantity_ordered,total_order_value,avg_unit_price,total_quantity_delivered,total_quantity_shipped,delivery_fulfillment_pct,on_time_shipments,total_shipments,on_time_rate_pct
0,MAT001,None,7,14000.0,700000.0,50.0,0.0,14000.0,0.0,2,7,28.57
1,MAT003,None,6,6250.0,250000.0,40.0,0.0,6420.0,0.0,4,5,80.00
2,MAT002,None,5,6000.0,180000.0,30.0,0.0,6000.0,0.0,1,5,20.00
3,MAT004,None,5,6880.0,172000.0,25.0,0.0,7500.0,0.0,3,4,75.00


## Cell 12 — Power BI Connection Guide

### Connect Power BI Desktop to MySQL
1. Open **Power BI Desktop**
2. **Home -> Get Data -> MySQL Database**
3. Server: `127.0.0.1`  |  Database: `shopx_dw`  ->  OK
4. Username: `root`  |  Password: your MySQL password  ->  Connect
5. Select the tables below and click **Load**

### Tables to Import (fact & dimension tables only)

| Table | Purpose |
|---|---|
| `fact_orders` | Orders with order_processing_days (now fully populated) |
| `fact_shipments` | Shipments with on_time_delivery, carrier info |
| `delivery_analytics` | KPI table — all 22 orders with processing/cycle/delay data |
| `order_items` | Item-level order detail |
| `dim_customers` | Customer master |
| `dim_carriers` | Carrier master |
| `dim_date` | Date dimension |

### Relationships to create in Model View

| From | Field | To | Field | Type |
|---|---|---|---|---|
| dim_customers | customer_id | fact_orders | customer_id | 1:Many |
| dim_customers | customer_id | fact_shipments | customer_id | 1:Many |
| dim_customers | customer_id | CustomerSatisfaction | customer_id | 1:Many |
| dim_carriers | carrier_id | fact_shipments | carrier_id | 1:Many |
| fact_orders | order_id | fact_shipments | order_id | 1:Many |

### Key DAX Measures

```dax
Total Orders          = COUNTROWS(fact_orders)
Total Revenue         = SUM(fact_orders[total_order_value])
Avg Processing Days   = AVERAGE(delivery_analytics[order_processing_days])
On-Time Rate %        = DIVIDE(COUNTROWS(FILTER(fact_shipments, fact_shipments[on_time_delivery]=1)), COUNTROWS(fact_shipments), 0) * 100
Avg Fulfillment Rate  = AVERAGE(delivery_analytics[fulfillment_rate_pct])
Avg CSAT Score        = AVERAGE(CustomerSatisfaction[csat_score])
Delivered Orders      = COUNTROWS(FILTER(fact_shipments, fact_shipments[shipment_status]="Delivered"))
In Transit Orders     = COUNTROWS(FILTER(fact_shipments, fact_shipments[shipment_status]="In Transit"))
```

### Expected KPI Values (for verification)
- Total Orders: 22
- Total Revenue: $229,000
- Avg Processing Days: ~7.2 days
- On-Time Delivery Rate: 45%
- Avg Fulfillment Rate: 100%
- Total Customers: 22
- Total Shipments: 20